# Execucao do pipeline

Detecção de anomalias em transações de Bitcoin, 2020. Este notebook **executa** as
39 etapas do repositório na ordem numérica, do `01` ao `39`, e grava um manifesto
com o custo e a duração reais de cada job.

> **Este notebook gasta dinheiro e leva dezenas de minutos.** Ele move o ano de 2020
> inteiro entre o BigQuery e o Cloud Storage, materializa as camadas silver e gold e
> treina sete modelos. Rode-o na véspera, nunca durante a apresentação.
>
> Para apresentar, use [`01-apresentacao.ipynb`](01-apresentacao.ipynb), que é
> estruturalmente incapaz de escrever em produção.

O notebook não contém SQL. Ele lê os 39 arquivos do repositório, resolve os
marcadores de ambiente com os valores de `config.env` e submete cada arquivo ao
BigQuery. Os arquivos `.sql` continuam sendo a fonte única de verdade.

## 1. Ambiente

In [ ]:
# Em BigQuery Studio (Colab Enterprise) as duas bibliotecas ja vem instaladas.
# Localmente:  pip install google-cloud-bigquery pandas
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd()))
import pipeline_lib as pl

cfg = pl.carregar_config()
print("Projeto :", cfg["PROJECT_ID"])
print("Bucket  :", "gs://" + cfg["BUCKET"])
print("Local   :", cfg["LOCATION"])

In [ ]:
from google.cloud import bigquery

# Localmente, autentique antes:  gcloud auth application-default login
cliente = bigquery.Client(project=cfg["PROJECT_ID"], location=cfg["LOCATION"])

# As queries referenciam `bronze.tabela` sem qualificar o projeto. Quem resolve o
# prefixo e o projeto padrao do cliente, o que mantem o SQL identico ao que roda
# colado no console do BigQuery.
print("cliente pronto em", cliente.project, "/", cliente.location)

## 2. Plano

`INICIAR_EM` e `PARAR_EM` delimitam a faixa a executar. Servem para retomar um
pipeline interrompido sem repetir o que já custou: se a execução falhou na etapa
20, corrija e recomece de `INICIAR_EM = 20`.

In [ ]:
INICIAR_EM = 1     # primeira etapa a executar (inclusive)
PARAR_EM   = 39    # ultima etapa a executar (inclusive)

etapas = pl.descobrir_etapas()
plano = [e for e in etapas if INICIAR_EM <= e.numero <= PARAR_EM]

import pandas as pd

pd.DataFrame([{
    "n": e.numero,
    "camada": e.camada,
    "arquivo": e.arquivo,
    "verbo": e.verbo,
    "escreve": "sim" if e.escreve else "-",
} for e in plano]).set_index("n")

## 3. Execução

Cada arquivo vira **um job** do BigQuery. Os arquivos com mais de uma instrução —
`06`, `12`, `29`, `30` — viram um job de script, com jobs filhos aninhados; o
somatório dos filhos é o custo real, e é ele que vai para o manifesto.

A execução é sequencial e **para no primeiro erro**. As dependências entre etapas
são rígidas: seguir depois de uma falha só produziria erros em cascata sobre uma
tabela que não existe.

In [ ]:
import time
from datetime import datetime, timezone


def estatisticas(job):
    """Custo real do job. Em scripts, soma os filhos, cujo pai nao contabiliza."""
    b, s = job.total_bytes_billed or 0, job.slot_millis or 0
    try:
        filhos = list(cliente.list_jobs(parent_job=job.job_id))
        if filhos:
            b = sum(f.total_bytes_billed or 0 for f in filhos)
            s = sum(f.slot_millis or 0 for f in filhos)
    except Exception:
        pass  # a listagem de filhos e um detalhe: nunca deve derrubar a execucao
    return b, s


def executar(etapa):
    sql = pl.resolver(etapa.sql_bruto(), cfg)
    t0 = time.time()
    job = cliente.query(sql)
    job.result()  # bloqueia ate o fim
    segundos = time.time() - t0
    b, s = estatisticas(job)
    return pl.Registro(
        numero=etapa.numero, camada=etapa.camada, arquivo=etapa.arquivo,
        verbo=etapa.verbo, status="ok", job_id=job.job_id,
        inicio=(job.started or datetime.now(timezone.utc)).isoformat(),
        fim=(job.ended or datetime.now(timezone.utc)).isoformat(),
        segundos=round(segundos, 2), bytes_faturados=b, slot_ms=s,
        linhas=job.num_dml_affected_rows,
    )

In [ ]:
registros = []
inicio_total = time.time()

for etapa in plano:
    print(f"[{etapa.numero:02d}] {etapa.camada}/{etapa.arquivo} ... ", end="", flush=True)
    try:
        reg = executar(etapa)
        registros.append(reg)
        print(f"ok  {pl.humanizar_segundos(reg.segundos):>12}  "
              f"{pl.humanizar_bytes(reg.bytes_faturados):>10}")
    except Exception as erro:
        registros.append(pl.Registro(
            numero=etapa.numero, camada=etapa.camada, arquivo=etapa.arquivo,
            verbo=etapa.verbo, status="erro", erro=str(erro)[:500],
        ))
        print("FALHOU")
        print(f"\n  {erro}\n")
        print("Execucao interrompida. Corrija e recomece de "
              f"INICIAR_EM = {etapa.numero}.")
        break

print(f"\nTotal: {pl.humanizar_segundos(time.time() - inicio_total)} "
      f"em {len(registros)} etapas")

## 4. Manifesto

O manifesto é o que liga este notebook ao da apresentação. Ele guarda `job_id`,
duração, bytes faturados e slot-milissegundos de cada etapa — os números que o
servidor cobrou, não os que o cliente cronometrou — e é a partir dele que a
apresentação desenha a linha do tempo do pipeline.

Ele fica fora do controle de versão (`notebooks/.gitignore`): é específico do
ambiente de quem executou.

In [ ]:
caminho = pl.gravar_manifesto(registros, cfg)
print("manifesto gravado em", caminho)

resumo = pd.DataFrame([{
    "n": r.numero, "camada": r.camada, "arquivo": r.arquivo, "status": r.status,
    "duracao": pl.humanizar_segundos(r.segundos),
    "faturado": pl.humanizar_bytes(r.bytes_faturados),
    "linhas": r.linhas,
} for r in registros]).set_index("n")

ok = [r for r in registros if r.status == "ok"]
print(f"\n{len(ok)} de {len(plano)} etapas concluidas | "
      f"total faturado: {pl.humanizar_bytes(sum(r.bytes_faturados or 0 for r in ok))} | "
      f"tempo somado: {pl.humanizar_segundos(sum(r.segundos or 0 for r in ok))}")
resumo

## 5. Depois daqui

Com o manifesto gravado e as tabelas materializadas, abra
[`01-apresentacao.ipynb`](01-apresentacao.ipynb). Ele não repete nada do que foi
feito aqui: lê os resultados, verifica o SQL pesado sem executá-lo, demonstra três
etapas ao vivo sobre o recorte de um dia e apresenta os resultados do ano completo.